# Preparação e concatenação dos dados — BPS (2020-2026)

**Sprint 2 — Preparação e concatenação das bases**

Este notebook trata os 7 arquivos brutos (confirmados como estruturalmente idênticos na Sprint 1) e gera o dataset consolidado `BPS_20_26_Waldinei.csv`.

In [5]:
import pandas as pd
from pathlib import Path

def encontrar_raiz_projeto(inicio: Path, marcador=("data", "raw")) -> Path:
    """Sobe pelos diretórios a partir de 'inicio' até achar uma pasta
    que contenha data/raw dentro dela — essa é a raiz do projeto."""
    atual = inicio.resolve()
    for pasta in [atual] + list(atual.parents):
        if (pasta.joinpath(*marcador)).exists():
            return pasta
    raise FileNotFoundError(f"Não encontrei 'data/raw' subindo a partir de {inicio}")

RAIZ_PROJETO = encontrar_raiz_projeto(Path.cwd())
PASTA_DADOS = RAIZ_PROJETO / "data" / "raw"

print("Raiz do projeto:", RAIZ_PROJETO)

Raiz do projeto: C:\Users\waldinei.rosa\OneDrive\Documentos\SCTEC\Modulo_2\Mini_Projeto\projeto_bps_waldinei


In [6]:
# Confirmação de que os 7 arquivos brutos estão presentes e com conteúdo
anos = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

for ano in anos:
    caminho = PASTA_DADOS / f"{ano}.csv"
    tamanho_mb = caminho.stat().st_size / (1024 * 1024)
    print(f"{ano}.csv: {tamanho_mb:.2f} MB")

2020.csv: 50.07 MB
2021.csv: 50.42 MB
2022.csv: 53.21 MB
2023.csv: 20.77 MB
2024.csv: 17.78 MB
2025.csv: 21.91 MB
2026.csv: 7.42 MB


In [7]:
df_2020 = pd.read_csv(PASTA_DADOS / "2020.csv", sep=";", encoding="utf-8")

print("Formato (linhas, colunas):", df_2020.shape)
print("\n--- Tipos de dados que o pandas detectou ---")
print(df_2020.dtypes)

Formato (linhas, colunas): (84919, 36)

--- Tipos de dados que o pandas detectou ---
ano_compra                int64
cnpj_instituicao          int64
sg_uf                    object
ds_esfera                object
dt_compra                object
dt_insercao              object
validade_compra           int64
co_catmat                 int64
ds_item                  object
co_pdm                  float64
co_grupo                float64
no_grupo                 object
co_classe               float64
no_classe                object
fg_generico              object
tp_compra                object
sg_unidade_medida        object
cnpj_fornecedor           int64
no_fornecedor            object
cnpj_fabricante           int64
no_fabricante            object
qt_medicamento            int64
ds_observacao            object
no_instituicao           object
no_municipio             object
un_medida_capacidade     object
no_pdm                   object
nu_processo_compra       object
nu_ata             

In [8]:
print("--- Contagem de nulos por coluna (2020) ---")
nulos = df_2020.isnull().sum()
nulos_percentual = (nulos / len(df_2020) * 100).round(2)

resumo_nulos = pd.DataFrame({"qtd_nulos": nulos, "pct_nulos": nulos_percentual})
resumo_nulos = resumo_nulos[resumo_nulos["qtd_nulos"] > 0].sort_values("qtd_nulos", ascending=False)

print(resumo_nulos)

print("\n--- Linhas duplicadas ---")
print("Total de duplicadas:", df_2020.duplicated().sum())

--- Contagem de nulos por coluna (2020) ---
                      qtd_nulos  pct_nulos
nu_ata                    84355      99.34
sg_unidade_medida         53513      63.02
vl_capacidade             53513      63.02
registro_anvisa           47781      56.27
fg_generico               47781      56.27
co_pdm                       90       0.11
no_classe                    90       0.11
co_classe                    90       0.11
no_grupo                     90       0.11
co_grupo                     90       0.11
no_pdm                       90       0.11
ds_observacao                32       0.04
un_medida_capacidade          6       0.01
un_fornecimento               6       0.01

--- Linhas duplicadas ---
Total de duplicadas: 0


In [9]:
linhas_sem_pdm = df_2020[df_2020["co_pdm"].isnull()]
print(linhas_sem_pdm[["ds_item", "no_grupo", "no_classe", "tp_compra"]].head(10))

                                                 ds_item no_grupo no_classe  \
22187  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
22563  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
22640  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
22888  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23090  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23253  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23284  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23396  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23399  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   
23632  DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA F...      NaN       NaN   

            tp_compra  
22187  ADMINISTRATIVA  
22563  ADMINISTRATIVA  
22640  ADMINISTRATIVA  
22888  ADMINISTRATIVA  
23090  ADM

In [10]:
print("Itens distintos entre as 90 linhas sem classificação:")
print(linhas_sem_pdm["ds_item"].value_counts())

print("\nTipos de compra dessas linhas:")
print(linhas_sem_pdm["tp_compra"].value_counts())

Itens distintos entre as 90 linhas sem classificação:
ds_item
DEXTROCETAMINA, CONCENTRAÇÃO:50 MG/ML, FORMA FARMACÊUTICA:SOLUÇÃO INJETÁVEL                                                                                                                                                                                                                                                                                                                                               30
CETAMINA CLORIDRATO, DOSAGEM:50 MG/ML, APLICAÇÃO:SOLUÇÃO INJETÁVEL                                                                                                                                                                                                                                                                                                                                                        28
CAMA HOSPITALAR, MATERIAL:AÇO INOXIDÁVEL, ACABAMENTO DA SUPERFÍCIE:PINTURA ELETROSTÁTICA, TIPO:3 MANIVELAS ESCAM

## Conclusão sobre nulos

As colunas com maior percentual de nulos (`nu_ata`, `sg_unidade_medida`, `vl_capacidade`, `registro_anvisa`, `fg_generico`) representam campos que não se aplicam a todos os tipos de item ou modalidade de compra — não são erros de preenchimento.

Já as colunas de classificação (`co_pdm`, `co_grupo`, `co_classe`, `no_pdm`, `no_grupo`, `no_classe`), vazias em 90 linhas (0,11%), têm uma causa identificável: **100% dessas linhas são do tipo de compra `ADMINISTRATIVA`**, sugerindo que esse tipo de aquisição não passa pela classificação padrão de materiais do BPS.

**Decisão:** os nulos serão mantidos como estão (não inferidos), e essa observação será documentada como limitação da base no README final.

In [11]:
print(df_2020[["dt_compra", "dt_insercao"]].head())
print("\nValores únicos de formato (amostra):")
print(df_2020["dt_compra"].sample(5, random_state=1).tolist())

    dt_compra dt_insercao
0  21/10/2020  11/09/2025
1  17/01/2020  25/05/2020
2  30/01/2020  03/02/2020
3  30/01/2020  03/02/2020
4  30/01/2020  03/02/2020

Valores únicos de formato (amostra):
['27/07/2020', '19/11/2020', '10/12/2020', '16/07/2020', '15/10/2020']


In [12]:
df_2020["dt_compra"] = pd.to_datetime(df_2020["dt_compra"], format="%d/%m/%Y", errors="coerce")
df_2020["dt_insercao"] = pd.to_datetime(df_2020["dt_insercao"], format="%d/%m/%Y", errors="coerce")

print("Datas que falharam na conversão (viraram NaT):")
print("dt_compra:", df_2020["dt_compra"].isnull().sum())
print("dt_insercao:", df_2020["dt_insercao"].isnull().sum())

print("\nIntervalo de dt_compra:", df_2020["dt_compra"].min(), "até", df_2020["dt_compra"].max())
print("Intervalo de dt_insercao:", df_2020["dt_insercao"].min(), "até", df_2020["dt_insercao"].max())

# Quantas linhas têm inserção muito depois da compra (mais de 1 ano de diferença)?
diferenca_dias = (df_2020["dt_insercao"] - df_2020["dt_compra"]).dt.days
print("\nLinhas com inserção mais de 365 dias após a compra:", (diferenca_dias > 365).sum())
print("Maior defasagem encontrada (dias):", diferenca_dias.max())

Datas que falharam na conversão (viraram NaT):
dt_compra: 0
dt_insercao: 0

Intervalo de dt_compra: 2020-01-01 00:00:00 até 2020-12-31 00:00:00
Intervalo de dt_insercao: 2020-01-02 00:00:00 até 2026-07-28 00:00:00

Linhas com inserção mais de 365 dias após a compra: 11952
Maior defasagem encontrada (dias): 2275


## Observação sobre datas: dt_compra vs dt_insercao

`dt_compra` está sempre corretamente contida no ano do arquivo. Já `dt_insercao` (data de carregamento no sistema) frequentemente ocorre muito depois da compra: 14% dos registros de 2020 têm mais de 1 ano de defasagem, com casos de até 6 anos.

**Decisão:** toda análise temporal do dashboard (evolução de valores, filtros por período) usará `dt_compra` como referência. `dt_insercao` é tratada apenas como metadado administrativo.

In [13]:
# CNPJ tem 14 dígitos. Se algum vier como número com menos de 14 caracteres
# quando convertido pra texto, é sinal de que perdeu zero à esquerda.
for coluna in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
    tamanhos = df_2020[coluna].astype(str).str.len()
    print(f"{coluna}: tamanhos únicos encontrados -> {sorted(tamanhos.unique())}")

cnpj_instituicao: tamanhos únicos encontrados -> [np.int64(12), np.int64(13), np.int64(14)]
cnpj_fornecedor: tamanhos únicos encontrados -> [np.int64(11), np.int64(12), np.int64(13), np.int64(14)]
cnpj_fabricante: tamanhos únicos encontrados -> [np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]


In [14]:
for coluna in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
    df_2020[coluna] = df_2020[coluna].astype(str).str.zfill(14)

# Confirma que agora todos têm exatamente 14 caracteres
for coluna in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
    tamanhos = df_2020[coluna].str.len()
    print(f"{coluna}: tamanhos únicos após correção -> {sorted(tamanhos.unique())}")

print("\nExemplo de CNPJ corrigido:")
print(df_2020[["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]].head())

cnpj_instituicao: tamanhos únicos após correção -> [np.int64(14)]
cnpj_fornecedor: tamanhos únicos após correção -> [np.int64(14)]
cnpj_fabricante: tamanhos únicos após correção -> [np.int64(14)]

Exemplo de CNPJ corrigido:
  cnpj_instituicao cnpj_fornecedor cnpj_fabricante
0   21467008000132  16970999000131  33408105000133
1   80905706000131  12889035000102  73856593000166
2   76105600000186  32743242000161  04459117000199
3   76105600000186  32743242000161  61286647000116
4   76105600000186  32743242000161  30222814000131


## Tratamento de CNPJs

Os CNPJs (`cnpj_instituicao`, `cnpj_fornecedor`, `cnpj_fabricante`) foram lidos originalmente como números inteiros, o que causou perda de zeros à esquerda (ex.: um CNPJ que deveria ter 14 dígitos aparecia com 10 a 13). Foram convertidos para texto e preenchidos com zeros à esquerda (`zfill(14)`) para restaurar o formato correto de 14 dígitos.

In [15]:
colunas_codigo = ["co_pdm", "co_grupo", "co_classe", "registro_anvisa"]

for coluna in colunas_codigo:
    df_2020[coluna] = df_2020[coluna].astype("Int64")

print(df_2020[colunas_codigo].dtypes)
print("\nExemplo de valores (incluindo os nulos que devem continuar como <NA>):")
print(df_2020[colunas_codigo].head(10))

co_pdm             Int64
co_grupo           Int64
co_classe          Int64
registro_anvisa    Int64
dtype: object

Exemplo de valores (incluindo os nulos que devem continuar como <NA>):
   co_pdm  co_grupo  co_classe  registro_anvisa
0   17708        65       6505             <NA>
1    3106        65       6505             <NA>
2    5116        65       6505             <NA>
3    5090        65       6505    1004704240082
4   10083        65       6505             <NA>
5    9702        65       6505    1018600030014
6   17807        65       6505             <NA>
7    5087        65       6505    1410701210026
8   17391        65       6505             <NA>
9    5130        65       6505    1384100230127


## Correção de tipo: códigos inteiros anuláveis

`co_pdm`, `co_grupo`, `co_classe` e `registro_anvisa` foram lidos pelo pandas como `float64` devido à presença de valores nulos (limitação do tipo `int64` tradicional, que não aceita nulos). Como são códigos identificadores — não quantidades a serem somadas ou calculadas — foram convertidos para `Int64` (tipo inteiro anulável do pandas), preservando tanto a natureza numérica quanto os nulos legítimos, sem o sufixo decimal `.0` indevido.

## Função de limpeza reutilizável

Com a lógica validada em 2020, a função abaixo encapsula todos os tratamentos decididos e será aplicada aos 7 anos.

In [16]:
def limpar_dataframe(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    """Aplica os tratamentos de limpeza validados no ano de 2020
    a qualquer DataFrame bruto do BPS."""
    df = df.copy()

    # Datas: texto -> datetime
    df["dt_compra"] = pd.to_datetime(df["dt_compra"], format="%d/%m/%Y", errors="coerce")
    df["dt_insercao"] = pd.to_datetime(df["dt_insercao"], format="%d/%m/%Y", errors="coerce")

    # CNPJs: número -> texto com 14 dígitos (evita perda de zero à esquerda)
    for coluna in ["cnpj_instituicao", "cnpj_fornecedor", "cnpj_fabricante"]:
        df[coluna] = df[coluna].astype(str).str.zfill(14)

    # Códigos identificadores: float -> Int64 (inteiro anulável)
    for coluna in ["co_pdm", "co_grupo", "co_classe", "registro_anvisa"]:
        df[coluna] = df[coluna].astype("Int64")

    # Rastreabilidade: marca de qual arquivo/ano cada linha veio
    df["ano_arquivo_origem"] = ano

    return df


# Teste rápido: aplica a função em 2020 de novo, a partir do df original,
# para confirmar que o resultado bate com o que fizemos manualmente
df_2020_bruto = pd.read_csv(PASTA_DADOS / "2020.csv", sep=";", encoding="utf-8")
df_2020_tratado = limpar_dataframe(df_2020_bruto, 2020)

print(df_2020_tratado.dtypes)
print("\nFormato:", df_2020_tratado.shape)

ano_compra                       int64
cnpj_instituicao                object
sg_uf                           object
ds_esfera                       object
dt_compra               datetime64[ns]
dt_insercao             datetime64[ns]
validade_compra                  int64
co_catmat                        int64
ds_item                         object
co_pdm                           Int64
co_grupo                         Int64
no_grupo                        object
co_classe                        Int64
no_classe                       object
fg_generico                     object
tp_compra                       object
sg_unidade_medida               object
cnpj_fornecedor                 object
no_fornecedor                   object
cnpj_fabricante                 object
no_fabricante                   object
qt_medicamento                   int64
ds_observacao                   object
no_instituicao                  object
no_municipio                    object
un_medida_capacidade     

## Aplicando a limpeza aos 7 anos e concatenando

Com a função validada, cada arquivo bruto é lido, tratado e empilhado num único DataFrame consolidado.

In [17]:
anos = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
dataframes_tratados = []

for ano in anos:
    df_bruto = pd.read_csv(PASTA_DADOS / f"{ano}.csv", sep=";", encoding="utf-8")
    df_tratado = limpar_dataframe(df_bruto, ano)
    dataframes_tratados.append(df_tratado)
    print(f"{ano}: {df_tratado.shape[0]} linhas, {df_tratado.shape[1]} colunas")

df_consolidado = pd.concat(dataframes_tratados, ignore_index=True)

print(f"\nTotal consolidado: {df_consolidado.shape[0]} linhas, {df_consolidado.shape[1]} colunas")
print(f"Soma das linhas individuais: {sum(df.shape[0] for df in dataframes_tratados)}")

2020: 84919 linhas, 37 colunas
2021: 85007 linhas, 37 colunas
2022: 89534 linhas, 37 colunas
2023: 33785 linhas, 37 colunas
2024: 28745 linhas, 37 colunas
2025: 34010 linhas, 37 colunas
2026: 11003 linhas, 37 colunas

Total consolidado: 367003 linhas, 37 colunas
Soma das linhas individuais: 367003


In [18]:
print("Duplicadas no consolidado:", df_consolidado.duplicated().sum())
print("\nIntervalo de ano_compra:", df_consolidado["ano_compra"].min(), "até", df_consolidado["ano_compra"].max())
print("\nContagem de linhas por ano_arquivo_origem:")
print(df_consolidado["ano_arquivo_origem"].value_counts().sort_index())

Duplicadas no consolidado: 0

Intervalo de ano_compra: 2020 até 2026

Contagem de linhas por ano_arquivo_origem:
ano_arquivo_origem
2020    84919
2021    85007
2022    89534
2023    33785
2024    28745
2025    34010
2026    11003
Name: count, dtype: int64


In [19]:
divergencias = df_consolidado[df_consolidado["ano_compra"] != df_consolidado["ano_arquivo_origem"]]
print("Linhas onde ano_compra difere do arquivo de origem:", len(divergencias))

if len(divergencias) > 0:
    print("\nExemplos:")
    print(divergencias[["ano_compra", "ano_arquivo_origem", "dt_compra", "dt_insercao"]].head(10))

Linhas onde ano_compra difere do arquivo de origem: 0


## Salvando o dataset consolidado

Com todas as validações concluídas, o dataset tratado é salvo em `data/processed/BPS_20_26_Waldinei.csv`.

In [ ]:
PASTA_PROCESSED = RAIZ_PROJETO / "data" / "processed"
PASTA_PROCESSED.mkdir(exist_ok=True)

caminho_saida = PASTA_PROCESSED / "BPS_20_26_Waldinei.csv"

df_consolidado.to_csv(caminho_saida, sep=";", index=False, encoding="utf-8")

tamanho_mb = caminho_saida.stat().st_size / (1024 * 1024)
print(f"Arquivo salvo em: {caminho_saida}")
print(f"Tamanho: {tamanho_mb:.2f} MB")
print(f"Linhas: {len(df_consolidado)}, Colunas: {len(df_consolidado.columns)}")